In [1]:
import pandas as pd, numpy as np
from scipy.spatial.distance import squareform, pdist, cdist
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path

In [2]:
def diffusion_map(data, eps, alpha=1, k=4):
    Dsq = squareform(pdist(data)**2)
    Wm = np.exp(-Dsq/eps); q = Wm.sum(1)
    Wa = Wm/np.outer(q**alpha, q**alpha)
    da = Wa.sum(1); Dis = 1.0/np.sqrt(da)
    S = Dis[:, None]*Wa*Dis[None, :]                
    w, v = np.linalg.eigh(S)
    idx = np.argsort(w)[::-1]; w, v = w[idx], v[:, idx]
    phi = Dis[:, None]*v                             
    Psi = phi[:, 1:k+1]*w[1:k+1]                     
    return {"evals": w, "Psi": Psi, "phi": phi, "W": Wm, "degrees": q}

def reconstruct_path(pred, start, end):
    path = [end]; current = end
    while current != start:
        current = pred[start, current]
        if current == -9999: return None
        path.append(current)
    return path[::-1]

def graph_degree_density(Psi, h):
    Dsq = squareform(pdist(Psi)**2)
    rho = np.exp(-Dsq/h).sum(1)
    scale = rho.mean(); rho /= scale
    rho = np.maximum(rho, 1e-6)
    return rho, -np.log(rho), scale

def density_and_bandwidth(Psi, multiplier=0.1):
    N = Psi.shape[0]
    median_Dsq = np.median(squareform(pdist(Psi)**2)[np.triu_indices(N,1)])
    h = multiplier * median_Dsq
    return h, median_Dsq, graph_degree_density(Psi, h)

def density_aware_cost(A_dist_sym, V, beta):
    rows, cols = A_dist_sym.nonzero()
    base = np.asarray(A_dist_sym[rows, cols]).ravel()
    costs = base*np.exp(beta*(V[rows]+V[cols])/2)
    return csr_matrix((costs, (rows, cols)), shape=A_dist_sym.shape)

def graph_path(A_dist_sym, beta, V, start, end):
    A = density_aware_cost(A_dist_sym, V, beta)
    _, pred = shortest_path(A, directed=False, return_predecessors=True)
    return reconstruct_path(pred, start, end)

def linear_path(Psi, start_idx, end_idx, n_grid=10):
    return np.linspace(Psi[start_idx], Psi[end_idx], num=n_grid)

def latent_density_at_points(gamma, Psi, h, scale):
    Dsq_query = cdist(gamma, Psi, metric="sqeuclidean")
    rho_query = np.exp(-Dsq_query / h).sum(axis=1)
    rho_query /= scale
    rho_query_floor = np.maximum(rho_query, 1e-6)
    V_query = -np.log(rho_query_floor)
    return rho_query_floor, V_query

def local_neighbourhood_lifting(Z, Psi, gamma, start_idx, end_idx, m=27, tau=0.2):
    nn = NearestNeighbors(n_neighbors=m); nn.fit(Psi)
    distances, indices = nn.kneighbors(gamma)
    a = np.exp(-distances**2/tau) / np.sum(np.exp(-distances**2 / tau), axis=1, keepdims=True)
    points = Z[indices, :]; z_hat = np.sum(a[:, :, None] * points, axis=1)
    z_hat[0], z_hat[-1] = Z[start_idx], Z[end_idx]
    return z_hat

def lift_validation(z_hat, Z):
    nn = NearestNeighbors(n_neighbors=1); nn.fit(Z)
    dNN, _ = nn.kneighbors(z_hat)
    return dNN

In [61]:
K_GRAPH = 15
BETA = 0.75

In [62]:
df = pd.read_parquet("./datasets/joint_df_quantile.parquet")
dates = df.index; variables = df.columns[:-1]
Z = df.to_numpy()[:, :-1]; N = Z.shape[0]

diff = diffusion_map(Z, eps=3, k=3); Psi = diff["Psi"]

endpoint_pairs = {"2006 benign -> 2008 GFC": ("2006-03-01", "2008-10-01"), "2019 benign -> 2020 COVID": ("2019-07-01", "2020-04-01"), 
                  "2019 benign -> 2021 Fiscal Tightening": ("2019-04-01", "2022-04-01"), "1977 benign -> 1982 Recession Trough": ("1977-01-01", "1982-07-01")}
endpoints = {name: (dates.get_loc(pair[0]), dates.get_loc(pair[1])) for name, pair in endpoint_pairs.items()}

A_dist = kneighbors_graph(Psi, n_neighbors=K_GRAPH, mode="distance", include_self=False)
A_dist_sym = A_dist.maximum(A_dist.T)

h_dens, med_Dsq, (rho, V, scale) = density_and_bandwidth(Psi)
TAU = med_Dsq

DZ = squareform(pdist(Z)); np.fill_diagonal(DZ, np.inf)
d_Z = np.min(DZ, axis=1); d_Z_median = np.median(d_Z)

In [63]:
def latent_path_length(coords):
    diffs = np.diff(coords, axis=0)
    return np.linalg.norm(diffs, axis=1).sum()

def path_smoothness(coords):
    if len(coords) < 3:
        return 0.0
    second_diff = coords[2:] - 2 * coords[1:-1] + coords[:-2]
    return np.sum(np.linalg.norm(second_diff, axis=1) ** 2)

def density_weighted_cost(coords, V_path, beta):
    diffs = np.diff(coords, axis=0)
    edge_lengths = np.linalg.norm(diffs, axis=1)

    V_mid = 0.5 * (V_path[:-1] + V_path[1:])
    return np.sum(edge_lengths * np.exp(beta * V_mid))

def ambient_nn_distances(Z_path, Z_cloud):
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(Z_cloud)
    dNN, _ = nn.kneighbors(Z_path)
    return dNN.ravel()

def interior_values(x):
    x = np.asarray(x)
    if len(x) > 2:
        return x[1:-1]
    return x

In [75]:
path_records = []

for event, (start_idx, end_idx) in endpoints.items():
    gamma = linear_path(Psi, start_idx, end_idx, n_grid=40)
    rho_lin, V_lin = latent_density_at_points(gamma, Psi, h_dens, scale)
    Z_lin = local_neighbourhood_lifting(Z, Psi, gamma, start_idx, end_idx, m=27, tau=TAU)
    path_records.append({"event": event, "path_type": "linear", "coords": gamma, "density": rho_lin, "V": V_lin, "Z_lift": Z_lin})

    idx_graph = graph_path(A_dist_sym, beta=0.0, V=V, start=start_idx, end=end_idx)
    path_records.append({"event": event, "path_type": "ordinary graph", "coords": Psi[idx_graph],
                         "density": rho[idx_graph], "V": V[idx_graph],"Z_lift": Z[idx_graph], "node_indices": idx_graph})

    idx_aware = graph_path(A_dist_sym, beta=BETA, V=V, start=start_idx, end=end_idx)
    path_records.append({"event": event,"path_type": "density-aware graph", "coords": Psi[idx_aware],
                         "density": rho[idx_aware], "V": V[idx_aware], "Z_lift": Z[idx_aware], "node_indices": idx_aware})

In [76]:
rows = []

for rec in path_records:
    coords = rec["coords"]; density = rec["density"]
    V_path = rec["V"]; Z_lift = rec["Z_lift"]

    dNN = ambient_nn_distances(Z_lift, Z)
    density_int = interior_values(density)

    rows.append({
        "event": rec["event"],
        "path_type": rec["path_type"],
        "number_points": rec["coords"].shape[0],
        "interior_minimum_density": density_int.min(),
        "interior_mean_density": density_int.mean(),
        "latent_path_length": latent_path_length(coords),
        "density_weighted_cost": density_weighted_cost(coords, V_path, beta=BETA),
        "smoothness": path_smoothness(coords),
        "max_ambient_NN_distance": dNN.max(),
        "mean_ambient_NN_distance": dNN.mean(),
        "mean_NN_observed_points": d_Z_median
    })

metrics_df = pd.DataFrame(rows)

In [77]:
requested_cols = ["event", "path_type", "number_points", "interior_minimum_density", "interior_mean_density", "latent_path_length",
                  "density_weighted_cost", "smoothness", "max_ambient_NN_distance", "mean_ambient_NN_distance"]

In [78]:
for event in metrics_df["event"].unique():
    event_table = (
        metrics_df[metrics_df["event"] == event]
        .set_index("path_type")
        .drop(columns="event")
    )
    print(event)
    display(event_table.round(4))

2006 benign -> 2008 GFC


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance,mean_NN_observed_points
path_type,,,,,,,,,
linear,40,0.0010,0.3546,1.0535,37.2737,0.0000,1.2766,1.0039,1.468
ordinary graph,9,0.1706,1.1336,1.0802,5.6283,0.3846,0.0000,0.0000,1.468
density-aware graph,9,0.3120,1.1213,1.0921,4.6787,0.4578,0.0000,0.0000,1.468


2019 benign -> 2020 COVID


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance,mean_NN_observed_points
path_type,,,,,,,,,
linear,40,0.0547,0.5079,0.5500,1.8478,0.0000,1.3133,1.1017,1.468
ordinary graph,8,0.1861,0.6669,0.5773,1.8074,0.0236,0.0000,0.0000,1.468
density-aware graph,9,0.2406,0.9343,0.5904,1.7349,0.0343,0.0000,0.0000,1.468


2019 benign -> 2021 Fiscal Tightening


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance,mean_NN_observed_points
path_type,,,,,,,,,
linear,40,0.4402,1.5913,0.3452,0.2843,0.0000,1.2246,1.1119,1.468
ordinary graph,9,0.8514,1.6905,0.3765,0.3345,0.0063,0.0000,0.0000,1.468
density-aware graph,10,0.9182,1.7466,0.3819,0.3137,0.0056,0.0000,0.0000,1.468


1977 benign -> 1982 Recession Trough


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance,mean_NN_observed_points
path_type,,,,,,,,,
linear,40,0.1379,0.6405,0.3366,0.6412,0.0000,1.1979,1.0573,1.468
ordinary graph,7,0.3444,0.9372,0.3689,0.6611,0.0079,0.0000,0.0000,1.468
density-aware graph,7,0.3444,0.9372,0.3689,0.6611,0.0079,0.0000,0.0000,1.468


## Results:

We get our main result that $M_\text{min}(\Gamma_\beta) > M_\text{min}(\gamma)$ reliably for all scenarios and for $\beta \in \{0.75, 1.5\}$

#### number_points
The number of points for the density-aware paths is higher than that for the standard graph path in two out of four cases here. Might be random, or might be the fact that the density-aware path needs to take longer routes through dense regions to avoid the penalty of a low density node.

Increasing $\beta$ from 0.75 to 1.5, now the number of nodes in the density-aware path is greater than the number of nodes in the standard graph path in every scenario. This gives evidence to the idea that the in the density-aware regime, taking more smaller steps through higher-density regions can be beneficial despite increasing the number of nodes.

#### interior_minimum_density
I have taken the interior minimum density because it is likely that the shared stress coordinate would be the minimum density region for all of these paths.

In every case, the minimum density of a node in the density-aware path is greater that or equal to the minimum density of either of the other paths. In the case of equality, this is because the density-aware and standard graph path are the same. This is what we hoped to see and it shows that our density aware algorithm is doing what it should be. The minimum density for the linear path is always substantially lower than the min density of either graph path, this reflects the fact that a linear interpolation does not produce realistic scenarios.

The method I used to calculate the density of latent points finds the exact density at each point on the line. However, the way we lift this line to macro-space uses a linear combination of nearby latent coordinates. Maybe it would be wise to calculate the density of the interpolation using a linear combination of these points, using the same weighting.


#### interior_mean_density
Also taking the mean of the interior because including endpoints adds nothing to our comparison. 

Here we see that the mean density for the linear path is always lower than that of either graph path (substantially lower in the case of the 2006-2008 path).

A strange result is that the mean density of the density-aware path is not always less than or equal to the standard graph path. For the 2006-2008 shock the mean density of the standard graph path is actually higher that that for the density-aware path. I initially thought that this may be due to a different number of points or an incorrect weight matrix, but now I think that this might be due to the fact that the density ($\rho$) and the cost ($V=-\log(\rho)$) have a non-linear relationship. So minimising $V$ does not necessarily correspond to maximising the mean density along a path.

After increasing $\beta$ from 0.75 to 1.5, the results for the 2006-2008 paths have changed. The density-aware path now has a higher mean density compared with the standard path. This shows that changing the penalty term $\beta$ can 'fix' what I saw before, however, it would be good to understand what is going on. 

#### latent_path_length
The linear path is always the shortest here by definition. The normal graph path is also always shorter than or equal to the density-aware path (also by definition).

#### density_weighted_cost
The density-weighted cost of the standard graph path is always greater than or equal to the cost for the density-aware path, this is also by construction.

What is strange to see is that the graph cost for the linear path is sometimes lower than both types of graph path, this should not be happening as it should be getting penalised for going through low density points. An explanation is that the linear paths are shorter than the graph paths, so even with a higher penalty, their cost could be lower. I am not convinced about this. There may be something wrong with either the cost function or the calculation of $V$ for the linear path.

Even after changing $\beta$ from 0.75 to 1.5, there is still a linear path that appears to have a lower density-weighted cost than both graph path costs 

#### smoothness
This is a metric which gives us a proxy of the curvature of the paths. It is a sum of the squared norms of the discrete second derivatives at each point in the path. A higher "smoothness" value means a path with higher curvature, or most jagged edges. I would have expected to see the density-aware paths to be more jagged as each hop is searching for the most dense neighbour rather than simply the closest one. Obviously, the linear paths have zero curvature.

The results are varied; in two cases the smoothness penalty for the density-aware path is higher than the standard graph path, but in one case the standard graph path has a lower smoothness penalty.

After changing $\beta$ from 0.75 to 1.5, the smoothness penalty of the density-aware paths are now all greater than the score for the standard graph path. This reflects the density-aware path taking a more drastic path to avoid low density regions.

#### *_NN_distance
In all cases, the maximum and the mean of the ambiant distance to the closes neighbour in macro-space from the lifted linear path is less than the mean distance between the observed variables in macro space. This means that the lifted coordinates are suitably realistic. In Phase 5, there is a more in depth table with percetiles further supporting the validity of this method.

### Investigating effect of $\beta$


In [ ]:
beta_path_records = []

for event, (start_idx, end_idx) in endpoints.items():
    for beta in [0.0, 0.75, 1.5, 2.5]:
        idx_graph = graph_path(A_dist_sym, beta=beta, V=V, start=start_idx, end=end_idx)
        beta_path_records.append({"beta": beta, "event": event, "coords": Psi[idx_graph], "density": rho[idx_graph],
                                  "V": V[idx_graph],"Z_lift": Z[idx_graph], "node_indices": idx_graph})

In [94]:
rows = []
for rec in beta_path_records:
    beta = rec["beta"]; coords = rec["coords"]
    density = rec["density"]; V_path = rec["V"]
    Z_lift = rec["Z_lift"]

    dNN = ambient_nn_distances(Z_lift, Z)
    density_int = interior_values(density)

    rows.append({
        "beta": beta,
        "event": rec["event"],
        "number_points": rec["coords"].shape[0],
        "interior_minimum_density": density_int.min(),
        "interior_mean_density": density_int.mean(),
        "latent_path_length": latent_path_length(coords),
        "density_weighted_cost": density_weighted_cost(coords, V_path, beta=2.5),
        "smoothness": path_smoothness(coords),
    })

beta_metrics_df = pd.DataFrame(rows)
for event in beta_metrics_df["event"].unique():
    event_table = (beta_metrics_df[beta_metrics_df["event"] == event].set_index("beta").drop(columns="event"))
    print(event)
    display(event_table.round(4))

2006 benign -> 2008 GFC


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness
beta,,,,,,
0.00,9,0.1706,1.1336,1.0802,532.6623,0.3846
0.75,9,0.3120,1.1213,1.0921,262.3223,0.4578
1.50,10,0.3120,1.3036,1.1211,262.3159,0.4589
2.50,11,0.3120,1.3948,1.1335,262.3154,0.4592


2019 benign -> 2020 COVID


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness
beta,,,,,,
0.00,8,0.1861,0.6669,0.5773,73.9534,0.0236
0.75,9,0.2406,0.9343,0.5904,60.0600,0.0343
1.50,9,0.2591,0.9639,0.6131,58.2086,0.0472
2.50,9,0.2591,1.0064,0.6254,58.2070,0.0489


2019 benign -> 2021 Fiscal Tightening


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness
beta,,,,,,
0.00,9,0.8514,1.6905,0.3765,0.4459,0.0063
0.75,10,0.9182,1.7466,0.3819,0.3528,0.0056
1.50,10,0.9474,1.7534,0.3837,0.3507,0.0064
2.50,11,0.9663,1.6938,0.3940,0.3388,0.0117


1977 benign -> 1982 Recession Trough


,number_points,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness
beta,,,,,,
0.00,7,0.3444,0.9372,0.3689,6.5059,0.0079
0.75,7,0.3444,0.9372,0.3689,6.5059,0.0079
1.50,9,0.3954,1.3601,0.4556,5.9201,0.0169
2.50,12,0.3954,1.6204,0.5164,5.9190,0.0186


It is clear that increasing $\beta$ does what we want it to do (increases mean and min density). We can also see that it does seem to increase the number of points in a path while increasing the smoothness score (making the path more jagged) apart from for 2019-2021 and $\beta=0.0 \rightarrow \beta=0.75$.

It is also worth noting that while the penalty is calculated for $\beta=2.5$, any value of $\beta$ seems to do quite a good job at reducing the path cost.

### Checking cost per unit length
This is to try to understand whether the cost function is working correctly for the linear paths

In [99]:
cost_df = metrics_df[["event", "path_type", "latent_path_length", "density_weighted_cost"]].copy()
cost_df["cost_per_length"] = cost_df["density_weighted_cost"] / cost_df["latent_path_length"]

for event in cost_df["event"].unique():
    event_table = (cost_df[cost_df["event"] == event]).set_index("path_type").drop("event", axis=1)
    print(event)
    display(event_table)


2006 benign -> 2008 GFC


,latent_path_length,density_weighted_cost,cost_per_length
path_type,,,
linear,1.053480,37.273723,35.381520
ordinary graph,1.080167,5.628309,5.210592
density-aware graph,1.092102,4.678701,4.284124


2019 benign -> 2020 COVID


,latent_path_length,density_weighted_cost,cost_per_length
path_type,,,
linear,0.550026,1.847779,3.359442
ordinary graph,0.577272,1.807401,3.130936
density-aware graph,0.590400,1.734907,2.938531


2019 benign -> 2021 Fiscal Tightening


,latent_path_length,density_weighted_cost,cost_per_length
path_type,,,
linear,0.345207,0.284343,0.823688
ordinary graph,0.376453,0.334459,0.888449
density-aware graph,0.381871,0.313716,0.821523


1977 benign -> 1982 Recession Trough


,latent_path_length,density_weighted_cost,cost_per_length
path_type,,,
linear,0.336583,0.641232,1.905121
ordinary graph,0.368926,0.661120,1.792012
density-aware graph,0.368926,0.661120,1.792012


The explanation that the cost was reduced by a very low length of a linear path is now no longer valid because the cost per length metric is still the lower for the linear path than the ordinary graph path in the 2019->2021 scenario.

Looking at the plot for this scenario, it does seem that the linear path goes straight throuh a very dense region, whereas the graph path detours to less dense regions. So maybe in this case, the linear path is genuinely a realistic path.